### Beam filters

In [3]:
import apache_beam as beam

# Output PCollection
class Output(beam.PTransform):
    class _OutputFn(beam.DoFn):
        def __init__(self, prefix=''):
            super().__init__()
            self.prefix = prefix

        def process(self, element):
            print(self.prefix+str(element))
            yield element

    def __init__(self, label=None,prefix=''):
        super().__init__(label)
        self.prefix = prefix

    def expand(self, input):
       return input | beam.ParDo(self._OutputFn(self.prefix))

In [4]:
with beam.Pipeline() as p: 
    value = (p | beam.Create(range(1, 11))
    | beam.Filter(lambda num: num%2 == 0)
    | "filtered output" >> Output(prefix='PCollection filtered value: ')
    | beam.Map(lambda num: num * 2)
    | "doubled output" >> Output(prefix='PCollection double value: '))
    ## NOTE: how to get a list and is it encourage?

    """
        Due to the distributed system foundation of Beam runner, it's not encouraged to collect/accumlate a PCollection in a local variable such as a list.
    If you have to you can do it vai a `beam.DoFn` via `beam.ParDo` or a PTransform encapsulating a `beam.DoFn`. Either way, it's still discouraged and even
    if you get to work with the local "DirectRunner" you're unlikely to get it to work in a real-life setup where each runner instance maintains it's own memory. 

        Moreover, this defeats the purpose of using Beam types such as PCollection.
    """


print("PValue: ", value)

PCollection filtered value: 2
PCollection double value: 4
PCollection filtered value: 4
PCollection double value: 8
PCollection filtered value: 6
PCollection double value: 12
PCollection filtered value: 8
PCollection double value: 16
PCollection filtered value: 10
PCollection double value: 20
PValue:  PCollection[[5]: doubled output/ParDo(_OutputFn).None]


In [5]:
# Filtering with multiple arguments.

import apache_beam as beam

def has_duration(plant, duration):
  return plant['duration'] == duration

with beam.Pipeline() as p:
  perennials = (
      p | 'Gardening plants' >> beam.Create([
          {
              'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'
          },
          {
              'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'
          },
          {
              'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'
          },
          {
              'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'
          },
          {
              'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'
          },
      ])
      # | 'Filter perennials' >> beam.Filter(has_duration, 'perennial')
      | 'Filter pernnials' >> beam.Filter(lambda plant, duration: plant['duration'] == duration, 'perennial')
      | beam.Map(print))

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


In [6]:
# Filtering with input as Singleton

import apache_beam as beam

with beam.Pipeline() as p:
  perennial = p | 'Perennial' >> beam.Create(['perennial'])

  perennials = (
      p | 'Gardening plants' >> beam.Create([
          {
              'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'
          },
          {
              'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'
          },
          {
              'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'
          },
          {
              'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'
          },
          {
              'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'
          },
      ])
      | 'Filter perennials' >> beam.Filter(
          lambda plant,
          duration: plant['duration'] == duration,
          duration=beam.pvalue.AsSingleton(perennial),
      )
      | beam.Map(print))

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


In [7]:
# Filtering with side input as Iterators 

import apache_beam as beam

with beam.Pipeline() as p:
  valid_durations = p | 'Valid durations' >> beam.Create([
      'annual',
      'biennial',
      'perennial',
  ])

  valid_plants = (
      p | 'Gardening plants' >> beam.Create([
          {
              'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'
          },
          {
              'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'
          },
          {
              'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'
          },
          {
              'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'
          },
          {
              'icon': '🥔', 'name': 'Potato', 'duration': 'PERENNIAL'
          },
      ])
      | 'Filter valid plants' >> beam.Filter(
          lambda plant,
          valid_durations: plant['duration'] in valid_durations,
          valid_durations=beam.pvalue.AsIter(valid_durations),
      )
      | beam.Map(print))

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'}


In [8]:
# Filtering with side input as dictionaries

import apache_beam as beam

with beam.Pipeline() as p:
  keep_duration = p | 'Duration filters' >> beam.Create([
      ('annual', False),
      ('biennial', False),
      ('perennial', True),
  ])

  perennials = (
      p | 'Gardening plants' >> beam.Create([
          {
              'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'
          },
          {
              'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'
          },
          {
              'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'
          },
          {
              'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'
          },
          {
              'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'
          },
      ])
      | 'Filter plants by duration' >> beam.Filter(
          lambda plant,
          keep_duration: keep_duration[plant['duration']],
          keep_duration=beam.pvalue.AsDict(keep_duration),
      )
      | beam.Map(print))


{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


## Aggregations

In [9]:
# Count globally
import apache_beam as beam

with beam.Pipeline() as p:
    total_elements = (
    p | 'Create plants' >> beam.Create(['🍓', '🥕', '🥕', '🥕', '🍆', '🍆', '🍅', '🍅', '🍅', '🌽'])
    | 'Count all elements' >> beam.combiners.Count.Globally()
    | beam.Map(print))

10


In [10]:
# Count per key 
import apache_beam as beam

with beam.Pipeline() as p:
  total_elements_per_keys = (
      p | 'Create plants' >> beam.Create([
          ('spring', '🍓'),
          ('spring', '🥕'),
          ('summer', '🥕'),
          ('fall', '🥕'),
          ('spring', '🍆'),
          ('winter', '🍆'),
          ('spring', '🍅'),
          ('summer', '🍅'),
          ('fall', '🍅'),
          ('summer', '🌽'),])
      | 'Count elements per key' >> beam.combiners.Count.PerKey()
      | beam.Map(print))


('spring', 4)
('summer', 3)
('fall', 2)
('winter', 1)


In [11]:
# Count unique elements

import apache_beam as beam

with beam.Pipeline() as p:
  total_unique_elements = (
      p | 'Create produce' >> beam.Create(['🍓', '🥕', '🥕', '🥕', '🍆', '🍆', '🍅', '🍅', '🍅', '🌽'])
      | 'Count unique elements' >> beam.combiners.Count.PerElement()
      | beam.Map(print))

('🍓', 1)
('🥕', 3)
('🍆', 2)
('🍅', 3)
('🌽', 1)


In [12]:
# Playground exercise

import apache_beam as beam

with beam.Pipeline() as p:
    _ = (p | beam.Create([(1, 36), (2, 91), (3, 33), (3, 11), (4, 67),])
    | beam.combiners.Count.PerKey()
    | beam.Map(print))

(1, 1)
(2, 1)
(3, 2)
(4, 1)


In [13]:
class SplitWords(beam.DoFn):
  def __init__(self, delimiter=' '):
    self.delimiter = delimiter

  def process(self, text):
    for word in text.split(self.delimiter):
      yield word

with beam.Pipeline() as p:
  _ = (p | beam.Create(["To be, or not to be: that is the question: Whether 'tis nobler in the mind to suffer, the slings and arrows of outrageous fortune, or to take arms against a sea of troubles, and by opposing end them. To die: to sleep"])
  | beam.ParDo(SplitWords()) | beam.combiners.Count.PerElement()
  # In order to sort the entries in the PCollection we can use TopN or collect all elements from all worder into a single list for small datasets using `beam.combiners.ToList()`
  # However, both cases do not scale well
  | beam.combiners.Top.Of(100, key=lambda kv: kv[1])
  | beam.FlatMap(lambda x: x)
  | Output(prefix='PCollection word count: '))

PCollection word count: ('to', 4)
PCollection word count: ('the', 3)
PCollection word count: ('To', 2)
PCollection word count: ('or', 2)
PCollection word count: ('of', 2)
PCollection word count: ('and', 2)
PCollection word count: ('sleep', 1)
PCollection word count: ('die:', 1)
PCollection word count: ('end', 1)
PCollection word count: ('opposing', 1)
PCollection word count: ('by', 1)
PCollection word count: ('troubles,', 1)
PCollection word count: ('sea', 1)
PCollection word count: ('a', 1)
PCollection word count: ('against', 1)
PCollection word count: ('take', 1)
PCollection word count: ('fortune,', 1)
PCollection word count: ('slings', 1)
PCollection word count: ('them.', 1)
PCollection word count: ('mind', 1)
PCollection word count: ('in', 1)
PCollection word count: ('nobler', 1)
PCollection word count: ('arms', 1)
PCollection word count: ('Whether', 1)
PCollection word count: ('outrageous', 1)
PCollection word count: ('arrows', 1)
PCollection word count: ('suffer,', 1)
PCollection

In [ ]:

import apache_beam as beam
import statistics

with beam.Pipeline() as p:
  total = (
      p | 'Create numbers' >> beam.Create([3, 4, 1, 2])
      | 'Sum values' >> beam.CombineGlobally(sum)
      # | 'Mean values' >> beam.CombineGlobally(statistics.mean)
      # | 'Mean values' >> beam.combiners.Mean.Globally()
      # NOTE: How does these 2 approaches for calculating mean differ and which one is recommended?
      # CombineGlobally collects all the data to one node and calculates the mean, therefore, using 
      # `beam.combiners.Mean.Globally()` is more optimal since it uses a 2 phase approach 
      # | 'Min value' >> beam.CombineGlobally(min) 
      | beam.Map(print))

1


In [17]:
# Sum per key 

import apache_beam as beam

with beam.Pipeline() as p:
  totals_per_key = (
      p | 'Create produce' >> beam.Create([
          ('🥕', 3),
          ('🥕', 2),
          ('🍆', 1),
          ('🍅', 4),
          ('🍅', 5),
          ('🍅', 3),])
      | 'Sum values per key' >> beam.CombinePerKey(sum)
      | beam.Map(print))

('🥕', 5)
('🍆', 1)
('🍅', 12)


In [6]:
# with keys example 

import apache_beam as beam

with beam.Pipeline() as p:
  (p | beam.Create(['apple', 'banana', 'cherry', 'durian', 'guava', 'melon'])
     | beam.WithKeys(lambda word: word[0:1])
     | beam.Map(print)
     )

('a', 'apple')
('b', 'banana')
('c', 'cherry')
('d', 'durian')
('g', 'guava')
('m', 'melon')
